# 02 - ERA5 Climate Data Analysis

This notebook explores ERA5 climate reanalysis data for temporal feature engineering.

## Objectives
- Understand ERA5 variable distributions and patterns
- Analyze seasonal and interannual variability
- Develop temporal aggregation strategies
- Identify derived climate features for soil modeling

## Data Source
- ERA5: https://cds.climate.copernicus.eu/
- 0.25° spatial resolution
- Hourly temporal resolution (aggregated to daily)

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

try:
    import xarray as xr
except ImportError:
    print("xarray not installed. Run: pip install xarray netCDF4")
    xr = None

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = Path('../../data/raw/era5')
RESULTS_DIR = Path('../../results/data_analysis')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load ERA5 Data

In [ ]:
# Load ERA5 dataset
era5_path = DATA_DIR / 'era5_daily.nc'

if era5_path.exists() and xr is not None:
    ds = xr.open_dataset(era5_path)
    print("ERA5 Dataset:")
    print(ds)
else:
    print(f"ERA5 data not found at {era5_path}")
    print("Run data/era5_downloader.py to download data")
    ds = None

In [ ]:
# Variable descriptions
VARIABLE_INFO = {
    't2m': {'name': '2m Temperature', 'unit': 'K', 'soil_relevance': 'Mineralization rate'},
    'tp': {'name': 'Total Precipitation', 'unit': 'm', 'soil_relevance': 'Leaching, moisture'},
    'stl1': {'name': 'Soil Temperature L1', 'unit': 'K', 'soil_relevance': 'Biological activity'},
    'swvl1': {'name': 'Soil Water L1', 'unit': 'm³/m³', 'soil_relevance': 'Nutrient mobility'},
    'ssrd': {'name': 'Solar Radiation', 'unit': 'J/m²', 'soil_relevance': 'Evapotranspiration'},
}

print("ERA5 Variables for Soil Modeling:")
for var, info in VARIABLE_INFO.items():
    print(f"  {var}: {info['name']} ({info['unit']}) - {info['soil_relevance']}")

## 2. Temporal Patterns

In [ ]:
# Sample location for time series analysis
SAMPLE_LAT = 40.0  # Central US latitude
SAMPLE_LON = -95.0  # Central US longitude

if ds is not None:
    # Extract time series at sample location
    point = ds.sel(latitude=SAMPLE_LAT, longitude=SAMPLE_LON, method='nearest')
    
    # Plot temperature time series
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    if 't2m' in point.data_vars:
        temp_c = point['t2m'] - 273.15  # Convert K to C
        axes[0].plot(temp_c.time, temp_c.values, alpha=0.7)
        axes[0].set_ylabel('Temperature (°C)')
        axes[0].set_title('2m Temperature Time Series')
    
    if 'tp' in point.data_vars:
        precip_mm = point['tp'] * 1000  # Convert m to mm
        axes[1].bar(precip_mm.time, precip_mm.values, alpha=0.7, width=1)
        axes[1].set_ylabel('Precipitation (mm)')
        axes[1].set_title('Daily Precipitation')
    
    if 'swvl1' in point.data_vars:
        axes[2].plot(point['swvl1'].time, point['swvl1'].values, alpha=0.7, color='brown')
        axes[2].set_ylabel('Soil Water (m³/m³)')
        axes[2].set_title('Soil Water Content Layer 1')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'era5_time_series.png', dpi=150)
    plt.show()

## 3. Seasonal Analysis

In [ ]:
# Monthly climatology
if ds is not None:
    point = ds.sel(latitude=SAMPLE_LAT, longitude=SAMPLE_LON, method='nearest')
    
    # Group by month
    monthly = point.groupby('time.month').mean()
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    months = np.arange(1, 13)
    month_names = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']
    
    if 't2m' in monthly.data_vars:
        axes[0, 0].bar(months, monthly['t2m'].values - 273.15, color='red', alpha=0.7)
        axes[0, 0].set_xticks(months)
        axes[0, 0].set_xticklabels(month_names)
        axes[0, 0].set_ylabel('Temperature (°C)')
        axes[0, 0].set_title('Mean Monthly Temperature')
    
    if 'tp' in monthly.data_vars:
        axes[0, 1].bar(months, monthly['tp'].values * 1000 * 30, color='blue', alpha=0.7)
        axes[0, 1].set_xticks(months)
        axes[0, 1].set_xticklabels(month_names)
        axes[0, 1].set_ylabel('Precipitation (mm/month)')
        axes[0, 1].set_title('Mean Monthly Precipitation')
    
    if 'swvl1' in monthly.data_vars:
        axes[1, 0].bar(months, monthly['swvl1'].values, color='brown', alpha=0.7)
        axes[1, 0].set_xticks(months)
        axes[1, 0].set_xticklabels(month_names)
        axes[1, 0].set_ylabel('Soil Water (m³/m³)')
        axes[1, 0].set_title('Mean Monthly Soil Moisture')
    
    if 'ssrd' in monthly.data_vars:
        axes[1, 1].bar(months, monthly['ssrd'].values / 1e6, color='orange', alpha=0.7)
        axes[1, 1].set_xticks(months)
        axes[1, 1].set_xticklabels(month_names)
        axes[1, 1].set_ylabel('Solar Radiation (MJ/m²)')
        axes[1, 1].set_title('Mean Monthly Solar Radiation')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'era5_seasonal_patterns.png', dpi=150)
    plt.show()

## 4. Derived Climate Features

Calculate agronomically-relevant derived features.

In [ ]:
def calculate_derived_features(ds, lat, lon):
    """Calculate derived climate features for a location."""
    point = ds.sel(latitude=lat, longitude=lon, method='nearest')
    
    features = {}
    
    if 't2m' in point.data_vars:
        temp_c = point['t2m'].values - 273.15
        
        # Growing Degree Days (base 10°C)
        gdd = np.maximum(temp_c - 10, 0)
        features['gdd_annual'] = gdd.sum()
        
        # Frost days
        features['frost_days'] = (temp_c < 0).sum()
        
        # Temperature extremes
        features['temp_mean'] = temp_c.mean()
        features['temp_range'] = temp_c.max() - temp_c.min()
        
    if 'tp' in point.data_vars:
        precip_mm = point['tp'].values * 1000
        
        # Annual precipitation
        features['precip_annual'] = precip_mm.sum()
        
        # Dry days
        features['dry_days'] = (precip_mm < 1).sum()
        
        # Extreme precipitation days
        features['heavy_precip_days'] = (precip_mm > 20).sum()
        
    if 'swvl1' in point.data_vars:
        soil_water = point['swvl1'].values
        
        # Soil moisture statistics
        features['soil_water_mean'] = soil_water.mean()
        features['soil_water_min'] = soil_water.min()
        features['drought_days'] = (soil_water < 0.2).sum()
    
    return features

# Calculate features for sample location
if ds is not None:
    features = calculate_derived_features(ds, SAMPLE_LAT, SAMPLE_LON)
    
    print("Derived Climate Features:")
    for key, value in features.items():
        print(f"  {key}: {value:.2f}")

## 5. Temporal Window Analysis

Evaluate different temporal aggregation windows for model input.

In [ ]:
# Different temporal windows to test
WINDOWS = [30, 90, 180, 365]  # days

print("Temporal Window Considerations:")
print("-" * 50)
print("")
print("30 days:  Recent weather, good for moisture-related predictions")
print("90 days:  Seasonal patterns, captures growing season dynamics")
print("180 days: Half-year, captures major seasonal transitions")
print("365 days: Full annual cycle, captures all seasonal variations")
print("")
print("Recommendation: Use 365-day window with positional encoding")
print("to capture full seasonal cycle while allowing model to learn")
print("which periods are most relevant for each soil property.")

In [ ]:
# Autocorrelation analysis
if ds is not None and 't2m' in ds.data_vars:
    point = ds.sel(latitude=SAMPLE_LAT, longitude=SAMPLE_LON, method='nearest')
    temp = point['t2m'].values
    
    # Calculate autocorrelation
    from scipy import signal
    
    # Detrend and normalize
    temp_norm = (temp - temp.mean()) / temp.std()
    
    # Autocorrelation
    autocorr = np.correlate(temp_norm, temp_norm, mode='full')
    autocorr = autocorr[len(autocorr)//2:]
    autocorr = autocorr / autocorr[0]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    lags = np.arange(0, min(400, len(autocorr)))
    ax.plot(lags, autocorr[:len(lags)])
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('Autocorrelation')
    ax.set_title('Temperature Autocorrelation')
    
    # Mark key periods
    for days, label in [(30, '30d'), (90, '90d'), (182, '6mo'), (365, '1yr')]:
        ax.axvline(x=days, color='gray', linestyle=':', alpha=0.7)
        ax.text(days, 0.9, label, ha='center')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'era5_autocorrelation.png', dpi=150)
    plt.show()

## 6. Summary and Recommendations

In [ ]:
# Generate summary
summary = {
    'data_source': 'ERA5 Reanalysis',
    'temporal_resolution': 'Daily',
    'spatial_resolution': '0.25 degrees',
    'recommended_variables': [
        't2m (2m temperature)',
        'tp (total precipitation)',
        'swvl1 (soil water content)',
        'stl1 (soil temperature)',
        'ssrd (solar radiation)'
    ],
    'derived_features': [
        'Growing Degree Days (GDD)',
        'Frost days count',
        'Precipitation extremes',
        'Drought indicators',
        'Temperature range'
    ],
    'recommendations': {
        'temporal_window': '365 days with positional encoding',
        'aggregation': 'Daily values with monthly summaries',
        'normalization': 'Z-score per variable globally'
    }
}

print("=" * 50)
print("ERA5 ANALYSIS SUMMARY")
print("=" * 50)
print(f"Resolution: {summary['spatial_resolution']} spatial, {summary['temporal_resolution']} temporal")
print("\nRecommended Variables:")
for var in summary['recommended_variables']:
    print(f"  - {var}")
print("\nDerived Features:")
for feat in summary['derived_features']:
    print(f"  - {feat}")
print("\nRecommendations:")
for key, value in summary['recommendations'].items():
    print(f"  - {key}: {value}")

In [ ]:
# Save summary
import json

with open(RESULTS_DIR / 'era5_patterns.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved to {RESULTS_DIR / 'era5_patterns.json'}")